# 한일합섬 ERP DB 탐색 노트북

`hhhs_db_manager` 를 노트북에서 쓰는 예시입니다. 위에서부터 순서대로 실행하고, 필요한 셀의 값(테이블명 · 조건)만 바꿔 쓰세요.

**준비**
1. 커널을 이 저장소의 `.venv` 로 선택합니다. 없으면 터미널에서 `uv venv && uv pip install -e ".[notebook]"`
2. 접속정보 `.env` 를 이 노트북과 같은 폴더에 둡니다 (README 「접속정보 설정」). 값은 팀 담당자에게 받습니다.

모든 결과는 pandas `DataFrame` 입니다. 셀 마지막 줄에 변수를 두면 표로 보입니다.
운영 서버라 **결과 10,000행 · 쿼리 60초 · 큰 테이블 조건 없는 조회 차단** 같은 보호 장치가 켜져 있습니다. 막히면 조건을 좁히는 것이 맞습니다.

In [ ]:
import pandas as pd
import hhhs_db_manager as db

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

db.check()   # 접속 · 권한 · 보호 설정 확인. 오류가 나면 README 「오류가 나면」 표를 보세요

## 1. 테이블 찾기

이름에 들어가는 문자열로 찾습니다. `rows` 는 서버 통계 행수라 즉시 나옵니다. 업무 테이블은 모두 `NEOE` 스키마에 있습니다.

In [ ]:
keyword = "SO"        # 테이블 이름에 들어갈 문자열 (대소문자 무시). 예: "QTIO", "PRQ", "ITEM", "PARTNER"

db.get_list_tables(keyword, min_rows=1)      # 빈 테이블 제외. 전체를 보려면 min_rows=None

## 2. 컬럼 보기

`name_kr` 은 ERP 안의 사전에서 붙인 한글명, `pk` 는 기본키 안의 순서입니다. 컬럼명은 영문 약어라 추측하지 말고 여기서 확인하세요.

In [ ]:
table = "SA_SOH"      # 스키마를 생략하면 NEOE. 다른 스키마는 "dbo.테이블" 처럼

db.get_columns(table)

## 3. 미리보기

`get_table` 은 `SELECT TOP {limit} … FROM 테이블` 을 만들어 줍니다. 조건 값은 `:이름` 으로 두고 키워드 인자로 넘깁니다(문자열을 직접 이어 붙이지 마세요).

In [ ]:
db.get_table(table, limit=20)

In [ ]:
db.get_table(
    "SA_SOL", limit=50,
    columns=["NO_SO", "SEQ_SO", "CD_ITEM", "QT_SO", "DT_DUEDATE", "STA_SO"],
    where="DT_DUEDATE >= :d AND CD_COMPANY = :co",     # 일자는 'YYYYMMDD' 문자열
    d="20260901", co="1000",                           # 운영 회사코드 1000
    order_by="DT_DUEDATE DESC",
)

## 4. 자유 SQL

규칙 세 가지 — 테이블 앞에 `NEOE.`, 일자는 `'YYYYMMDD'` 문자열 비교, 탐색은 `TOP N`. 값은 `:이름` 바인딩으로 넘깁니다.

In [ ]:
sql = """
SELECT TOP 20 H.NO_SO, H.DT_SO, H.CD_PARTNER, P.LN_PARTNER AS 거래처명, H.STA_SO
FROM NEOE.SA_SOH H
LEFT JOIN NEOE.MA_PARTNER P ON P.CD_COMPANY = H.CD_COMPANY AND P.CD_PARTNER = H.CD_PARTNER
WHERE H.CD_COMPANY = :co AND H.DT_SO >= :d
ORDER BY H.DT_SO DESC
"""
df = db.query(sql, co="1000", d="20260901")
df

## 5. 건수 · 집계

집계는 기간 조건을 넣고, 결과는 `COUNT(*)` 로 한 번 더 검산하는 습관을 들이세요.

In [ ]:
db.query("""
SELECT LEFT(DT_SO, 6) AS 월, COUNT(*) AS 수주건수
FROM NEOE.SA_SOH
WHERE CD_COMPANY = :co AND DT_SO >= :d
GROUP BY LEFT(DT_SO, 6)
ORDER BY 월
""", co="1000", d="20260101")

## 6. 코드값 뜻 찾기

`STA_SO = 'R'` 처럼 코드로 저장된 값의 뜻은 ERP 코드표(`MA_CODE` · `MA_CODEDTL`)에 있습니다. 필드의 한글명으로 찾습니다.

In [ ]:
def codes(field_name_kr):
    """ERP 코드표에서 값과 뜻을 찾는다. 예: codes("수주상태"), codes("작업상태"), codes("수불구분")"""
    return db.query("""
        SELECT c.CD_FIELD, c.NM_FIELD, d.CD_SYSDEF AS value, d.NM_SYSDEF AS meaning
        FROM NEOE.MA_CODEDTL d
        JOIN NEOE.MA_CODE c ON c.CD_FIELD = d.CD_FIELD AND c.CD_COMPANY = d.CD_COMPANY
        WHERE d.CD_COMPANY = 'MASTER' AND c.NM_FIELD LIKE :f AND d.USE_YN = 'Y'
        ORDER BY c.CD_FIELD, d.CD_SYSDEF
    """, f=f"%{field_name_kr}%")

codes("수주상태")

## 7. 결과 저장

결과 파일은 이 폴더에 두어도 git 에 올라가지 않습니다(`.gitignore`). 사내 자료이니 외부로 보내지 마세요.

In [ ]:
# df.to_csv("수주_최근.csv", index=False, encoding="utf-8-sig")   # 엑셀에서 한글 깨짐 없이 열림
# df.to_excel("수주_최근.xlsx", index=False)                       # openpyxl 필요: uv pip install openpyxl

## 막히면

| 메시지 | 조치 |
| --- | --- |
| `UserWarning: 결과가 10,000행에서 잘렸습니다` | 조건을 좁히거나 `db.query(sql, max_rows=50000)` |
| `QueryRejected: … 100만 행 이상인데 TOP · WHERE · 집계가 없습니다` | `WHERE` 에 기간 조건을 넣기 |
| `QueryTimeout` | 범위를 줄이기. 꼭 필요하면 `.env` 에 `HHHS_QUERY_TIMEOUT=300` |
| `DBError: TLS 협상 실패` | 커널 재시작 후 이 노트북의 첫 셀부터 다시 |
| 그 밖의 오류 | README 「오류가 나면」 표 |